# Multi-Turn Simulation with DeepEval ConversationSimulator

## Overview

This notebook demonstrates how to simulate multi-turn conversations using the [DeepEval simulation framework](https://deepeval.com/guides/guides-multi-turn-simulation). DeepEval generates synthetic conversations from predefined scenarios, enabling you to test your GenAI application's conversational behavior at scale without manual interaction.

## What You'll Learn

1. How to define simulation scenarios with `ConversationalGolden`.
2. How to run a basic simulation with `ConversationSimulator`.
3. How to control conversation length with `max_user_simulations`.
4. How to run simulations in parallel for faster execution.
5. How to use a custom simulator model (Amazon Nova Micro via Bedrock).
6. How to start simulations from existing turns.
7. How to use lifecycle hooks for per-scenario callbacks.

## Key Concepts

- **ConversationalGolden** — A test specification that defines *what* to simulate: a scenario description, a user persona, and an expected outcome. No scripted messages, just constraints.
- **ConversationSimulator** — Orchestrates the back-and-forth: generates realistic user messages turn by turn, sends each to your application via a callback, and repeats until the expected outcome is reached or a turn limit is hit.
- **ConversationalTestCase** — The resulting conversation (all user and assistant turns in order), packaged for inspection or scoring against multi-turn metrics.

# Setup

For this guide, we will use a GenAI travel booking assistant built with Strands Agents framework presented in the previous notebooks. The assistant handles the end-to-end workflow of planning and managing travel: searching for flights and hotels, placing bookings, retrieving existing reservations, and processing cancellations.

This application makes a strong evaluation target — it naturally demands multi-turn interaction. A user rarely searches, selects, and books in a single message. Instead, a typical session might span half a dozen turns: asking for flights, narrowing options by price, switching to hotel search, booking both, and then circling back to modify a reservation.

The architecture is intentionally simple. At its core, it is an LLM-powered conversational agent equipped with different tools that give it the ability to act on user requests. Booking state is stored in a simple in-memory dictionary.

For simplicity and consistency, search results are drawn from hardcoded mock data. This keeps the agent self-contained and deterministic on the tool side, ensuring that any variability in simulation results stems from the LLM's conversational behavior, not from external dependencies.

We start by installing the requirements for this project.

In [ ]:
# Install all required packages (deepeval, strands, boto3, etc.)
%pip install -r requirements.txt

The following code imports the required agent and tool components from Strands framework.

In [ ]:
# Import Strands Agent class and tool decorator for defining agent tools
from strands import Agent, tool
from datetime import date

Now, let's define the assistant's code. The cell below defines the in-memory booking state and six tools that the agent will use: `search_flights`, `search_hotels`, `book_flight`, `book_hotel`, `get_bookings`, and `cancel_booking`. Each tool returns a formatted string that the agent relays to the user.

In [ ]:
# --- In-memory booking state ---
bookings: dict = {}
booking_counter = 0


# --- Tool definitions: each @tool becomes available to the agent ---

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Search for available flights between two cities.

    Args:
        origin: Departure city or airport code (e.g. 'New York' or 'JFK')
        destination: Arrival city or airport code (e.g. 'London' or 'LHR')
        departure_date: Departure date in YYYY-MM-DD format
        return_date: Optional return date in YYYY-MM-DD format for round trips
    """
    flights = [
        {"flight": "AA101", "airline": "American Airlines", "departs": "08:00", "arrives": "20:00", "price": 450, "class": "Economy"},
        {"flight": "BA202", "airline": "British Airways",   "departs": "11:30", "arrives": "23:45", "price": 620, "class": "Economy"},
        {"flight": "UA303", "airline": "United Airlines",   "departs": "14:00", "arrives": "02:15", "price": 890, "class": "Business"},
    ]
    trip = f"round-trip (return: {return_date})" if return_date else "one-way"
    lines = [f"Flights from {origin} to {destination} on {departure_date} ({trip}):\n"]
    for f in flights:
        lines.append(
            f"  {f['flight']} | {f['airline']} | {f['departs']}–{f['arrives']} | "
            f"${f['price']} | {f['class']}"
        )
    return "\n".join(lines)


@tool
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Search for available hotels in a city.

    Args:
        city: Destination city
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
    """
    hotels = [
        {"name": "Grand Plaza Hotel",    "stars": 5, "price_per_night": 320, "amenities": "Pool, Spa, Restaurant"},
        {"name": "City Center Inn",       "stars": 3, "price_per_night": 95,  "amenities": "Free WiFi, Breakfast"},
        {"name": "Boutique Stay",         "stars": 4, "price_per_night": 180, "amenities": "Gym, Bar, Concierge"},
    ]
    nights = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    lines = [f"Hotels in {city} | {check_in} → {check_out} ({nights} nights, {guests} guest(s)):\n"]
    for h in hotels:
        total = h["price_per_night"] * nights
        lines.append(
            f"  {'★' * h['stars']} {h['name']} | ${h['price_per_night']}/night (total ${total}) | {h['amenities']}"
        )
    return "\n".join(lines)


@tool
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> str:
    """Book a flight for a passenger.

    Args:
        flight_number: Flight number to book (e.g. 'AA101')
        passenger_name: Full name of the passenger
        origin: Departure city or airport
        destination: Arrival city or airport
        travel_date: Travel date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"FLT-{booking_counter:04d}"
    bookings[ref] = {
        "type": "flight", "flight": flight_number, "passenger": passenger_name,
        "route": f"{origin} → {destination}", "date": travel_date, "status": "Confirmed"
    }
    return f"✅ Flight booked! Ref: {ref} | {flight_number} | {origin} → {destination} on {travel_date} | Passenger: {passenger_name}"


@tool
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.

    Args:
        hotel_name: Name of the hotel to book
        guest_name: Full name of the guest
        city: City where the hotel is located
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"HTL-{booking_counter:04d}"
    bookings[ref] = {
        "type": "hotel", "hotel": hotel_name, "guest": guest_name,
        "city": city, "check_in": check_in, "check_out": check_out, "status": "Confirmed"
    }
    return f"✅ Hotel booked! Ref: {ref} | {hotel_name} in {city} | {check_in} → {check_out} | Guest: {guest_name}"


@tool
def get_bookings() -> str:
    """Retrieve all current bookings."""
    if not bookings:
        return "No bookings found."
    lines = ["Current bookings:\n"]
    for ref, b in bookings.items():
        if b["type"] == "flight":
            lines.append(f"  [{ref}] ✈ {b['flight']} | {b['route']} on {b['date']} | {b['passenger']} | {b['status']}")
        else:
            lines.append(f"  [{ref}] 🏨 {b['hotel']} in {b['city']} | {b['check_in']} → {b['check_out']} | {b['guest']} | {b['status']}")
    return "\n".join(lines)


@tool
def cancel_booking(booking_ref: str) -> str:
    """Cancel an existing booking.

    Args:
        booking_ref: Booking reference number (e.g. 'FLT-0001' or 'HTL-0002')
    """
    if booking_ref not in bookings:
        return f"Booking {booking_ref} not found."
    bookings[booking_ref]["status"] = "Cancelled"
    return f"❌ Booking {booking_ref} has been cancelled."

# All tools return formatted strings so the agent can relay results to the user.

Now we initialize the Strands agent with a travel-assistant system prompt and all six tools defined above.

In [ ]:
# Initialize the travel-assistant agent with a system prompt and all booking tools
agent = Agent(
    system_prompt=(
        "You are a helpful travel booking assistant. You help users search for flights and hotels, "
        "make bookings, view existing reservations, and cancel bookings. "
        "Always confirm details with the user before completing a booking. "
        "Use today's date as context when dates are not fully specified."
    ),
    tools=[search_flights, search_hotels, book_flight, book_hotel, get_bookings, cancel_booking],
    callback_handler=None,  # suppress streaming output to avoid Rich/Jupyter recursion
)

Before diving into simulation, let's verify the agent works with a single-turn request that searches for flights and books the cheapest option.

In [ ]:
# Quick smoke test: search flights and book the cheapest in a single turn
agent("Search for flights from New York to London on 2025-09-01, then book the cheapest one for Jane Doe")

# DeepEval Multi-Turn Simulation Framework

Multi-turn simulation automatically generates realistic conversations between a simulated user and your LLM application. It is the foundation of multi-turn evaluation: without it, every scenario you want to test requires someone to manually chat with your application.

In single-turn evaluation, you can write a fixed list of inputs and expected outputs. Multi-turn conversations break this model because the user’s next message depends on what the assistant just said. Each turn branches the dialogue in a new direction, making it impossible to script exchanges in advance. Simulation solves this by having an LLM role-play as the user, generating contextually appropriate messages in real time while your actual application responds.

## Simulation vs. Traditional Testing

Teams that skip simulation typically rely on one of two approaches, both of which have serious limitations.

**Manual testing** involves a team member chatting with the application and reviewing the results by hand.
- Does not scale: testing 50 scenarios across multiple turns takes hours.
- Non-reproducible: different testers phrase messages differently, making before-and-after comparisons unreliable.
- Biased: human testers unconsciously steer conversations toward expected paths, missing the edge cases that real users routinely trigger.

**Historical replay** exports past conversations from production and evaluates them offline.
- Conversations were shaped by the responses of your *current* system.
- If you change a prompt or swap a model, the user would have responded differently at every turn.
- Old transcripts become irrelevant for evaluating the new version.

**Simulation** avoids both issues and provides various advantages compared to other approaches. 

| Advantage | Description |
|---|---|
| Reproducible | Same scenario and application version <br>produce the same (or statistically similar) <br>conversation every time. |
| Scalable | Generate hundreds of conversations <br>in parallel in minutes, not hours. |
| Forward-looking | Every simulation runs against your <br>current application, so regressions <br>surface immediately. |
| Diverse | The simulated user introduces natural <br>variation, exposing edge cases you would <br>not think to test manually. |

## Simulation Flow

The `ConversationSimulator` manages the entire conversation flow. It works through the following simulation cycle:


<img src="img/deepeval_sim.png" alt="DeepEval ConversationSimulator Flow" width="25%" style="border: 1px solid #ccc; border-radius: 6px; padding: 4px;">

1. The simulator reads the `scenario`, `user_description`, and `expected_outcome` from a `ConversationalGolden`.
2. Using LLM (configured via `simulator_model`), the simulator generates a realistic user message based on the persona and the full conversation history.
3. The message is passed to your application through the `model_callback` function.
4. The assistant's response is returned as a `Turn` object (which can include `content`, `tools_called`, and `retrieval_context`) and appended to the conversation history.
5. The simulator evaluates whether the `expected_outcome` has been satisfied. If not, and `max_user_simulations` has not been reached, it loops back to step 2.
6. Once the conversation concludes, the full exchange is bundled into a `ConversationalTestCase` — ready to be passed to `evaluate()` for scoring.

## Building Golden Scenarios

Rather than defining a fixed script of messages and expected replies, a `ConversationalGolden` captures the conditions under which a conversation should take place: a scenario description, a persona, and an expected outcome.

Think of it as a test specification: it tells the system *what* to test without prescribing *how* the conversation will unfold.

| Field | Purpose |
|---|---|
| `scenario` | Describes the conversational situation to simulate |
| `expected_outcome` | Defines what a successful resolution looks like |
| `user_description` | Sets the persona and behavioral traits <br>of the simulated user |

The simulator relies on all three fields to produce realistic user messages. The `scenario` defines the topic of conversation, the `user_description` influences the tone and behavior, and the `expected_outcome` signals when the conversation has naturally reached its end.

`ConversationalGolden` items can be versioned and stored in source control alongside your application code.

The next snippet defines two simulation scenarios.

In [ ]:
from deepeval.dataset import ConversationalGolden

# Define two evaluation scenarios with distinct user personas
goldens = [
    ConversationalGolden(
        scenario="A couple planning a round-trip vacation from San Francisco to Tokyo, looking for flights and hotel bundles within a set budget",
        expected_outcome="User selects a flight and hotel package that fits their budget and receives a confirmed itinerary with all trip details",
        user_description="Detail-oriented planner who compares multiple options and asks about cancellation policies before committing"
    ),
    ConversationalGolden(
        scenario="A business traveler needing to reschedule an upcoming flight from Chicago to Berlin due to a last-minute meeting conflict",
        expected_outcome="User successfully changes the flight to a new date, confirms the updated schedule, and understands any fare differences or fees applied",
        user_description="Time-sensitive professional who values quick resolutions and prefers minimal back-and-forth"
    )
]

print(f"Defined {len(goldens)} conversation scenarios.")

Note that a detailed `user_description` leads to a more realistic simulation. For example, a generic description like 'A traveler' produces bland interactions, while something like 'A first-time international traveler who is unfamiliar with visa requirements and frequently asks for reassurance about layover procedures' creates far more engaging and challenging conversations.

## The Model Callback

The `model_callback` serves as the connection point between the simulator and your application. It is an async function that takes in a user message and returns your application's response wrapped in a Turn object. This is not the case with our assistant agent.

Most applications, including the travelling assistant, require the full conversation history to produce contextually relevant responses. To enable this, add the `turns` parameter to the call. 

The `turns` parameter holds the complete record of all previous exchanges in the conversation, covering both user and assistant messages. This is particularly useful when your application manages conversation history directly rather than depending on an external session store, as in our case.

The `Turn` object supports more than just text content. If your application leverages a RAG pipeline or invokes tools, you may include those details in the returned turn so that specialized metrics can properly evaluate them. We are adding the response text and tools information in the turn. 

In [ ]:
from deepeval.test_case import Turn, ToolCall

def agent_callback(input: str) -> Turn:
    """Invoke the Strands agent and return a DeepEval Turn. 
       We use a local agent running in the notebook, however, in a real-world scenario you 
       would typically invoke a remote agent API for agent deplyed to production runtime, 
       such as the one provided by AgentCore runtime running the agent service."""

    response = agent(input)
    text = response.message["content"][0]["text"] if response.message.get("content") else str(response)
    tools = [ToolCall(name=tm.tool["name"], 
                      input_parameters=tm.tool.get("input")) 
             for tm in response.metrics.tool_metrics.values()] or None
    return Turn(role="assistant", 
                content=text,
                tools_called=tools)

The following fields are provided in `Turn`:

| Field | Type | What It Enables |
|---|---|---|
| `content` | `str` | Required by all metrics |
| `retrieval_context` | `List[str]` | `TurnFaithfulnessMetric`, <br>`TurnContextualRelevancyMetric`, <br>and other multi-turn RAG metrics |
| `tools_called` | `List[ToolCall]` | `ToolUseMetric`, <br>`GoalAccuracyMetric` |
| `additional_metadata` | `Dict` | Custom key-value pairs <br>for logging and debugging |

## Using Custom Model

The simulated user is powered by an LLM. The default model provider used by Deep Evals for this purpose is OpenAI. To replace it with Bedrock, we should replace this model with one of Bedrock models, amazon.nova-micro-v1:0. We implement this custom model by creating a class that inherits from `DeepEvalBaseLLM`.

In [ ]:
import json
import boto3
from deepeval.models import DeepEvalBaseLLM

class BedrockNovaMicro(DeepEvalBaseLLM):
    """DeepEval-compatible wrapper around Amazon Nova Micro via Bedrock.
    Used as the simulator and evaluation judge model instead of the default OpenAI backend."""

    MODEL_ID = "amazon.nova-micro-v1:0"

    def __init__(self, region: str = "us-east-1"):
        self.region = region
        super().__init__(model=self.MODEL_ID)

    def load_model(self):
        return boto3.client("bedrock-runtime", region_name=self.region)

    def get_model_name(self) -> str:
        return self.MODEL_ID

    def _invoke(self, prompt: str, schema=None) -> str:
        system_text = "You are a helpful assistant. Respond only in valid JSON." if schema else None
        if schema:
            json_schema = schema.model_json_schema()
            prompt = (
                f"{prompt}\n\n"
                f"Return ONLY valid JSON matching this schema (no markdown, no extra text):\n"
                f"{json.dumps(json_schema, indent=2)}"
            )

        messages = [{"role": "user", "content": [{"text": prompt}]}]
        body = {"messages": messages, "inferenceConfig": {"maxTokens": 4096, "temperature": 0.0}}
        if system_text:
            body["system"] = [{"text": system_text}]

        response = self.model.converse(modelId=self.MODEL_ID, **body)
        text = response["output"]["message"]["content"][0]["text"]

        if schema:
            try:
                return schema.model_validate_json(text)
            except Exception:
                data = json.loads(text)
                return schema.model_validate(data)
        return text

    def generate(self, prompt: str, schema=None) -> str:
        return self._invoke(prompt, schema)

    async def a_generate(self, prompt: str, schema=None) -> str:
        return self._invoke(prompt, schema)


# Instantiate the Bedrock model for use as simulator and evaluation judge
nova_micro = BedrockNovaMicro()
print(f"Model loaded: {nova_micro.get_model_name()}")

## Simulating conversations with ConversationSimulator

The `ConversationSimulator` manages the entire conversation flow. It works through the following simulation cycle:

1. Read the scenario and user_description from a ConversationalGolden
2. Generate a realistic user message based on the scenario and the conversation so far
3. Send the message to your application (through the `model_callback` callback, see below)
4. Capture the assistant's response
5. Evaluate whether the `expected_outcome` has been satisfied
7. Repeat steps 2 through 5 until the outcome is met or the maximum number of turns is reached

The simulator returns ConversationalTestCases — complete conversations with all turns recorded—ready for further processing and analysis, e.g., evaluations with multi-turn metrics.

By default, simulations run for up to 10 user-assistant cycles. You can adjust this with `max_user_simulations` parameter. Let's decrease or increase the number of cycles according to the contenxt - decrease to 5 or less for quick tests and increase to 20-30 for production tests.

We also supply our simulator with the custom Amazon Nova Micro model defined above.

In [ ]:

from deepeval.simulator import ConversationSimulator

simulator = ConversationSimulator(model_callback=agent_callback, 
                                  simulator_model=nova_micro)

test_cases = simulator.simulate(
    conversational_goldens=goldens,
    max_user_simulations=5    
)

print(f"\nSimulated {len(test_cases)} conversations.")
for i, conv in enumerate(test_cases):
    print(f"\n--- Conversation {i+1} ({len(conv.turns)} turns) ---")
    for turn in conv.turns:
        preview = turn.content[:120].replace("\n", " ")
        print(f"  [{turn.role}] {preview}..." if len(turn.content) > 120 else f"  [{turn.role}] {preview}")

### Starting in the Mid-Turn

Some applications start with a fixed opening message, such as a greeting or disclaimer. You can define these initial turns on the golden, and the simulator will pick up the conversation from that point. It recognizes the existing assistant turn and generates a user response that flows naturally from it. You may use this technique when:

1. Your application always opens with a greeting or disclaimer
2. You want to test how the application handles a mid-conversation handoff
3. You have a partially completed conversation that you want to continue

In [ ]:
goldens = [ConversationalGolden(
    scenario="Traveler asking about flight rebooking policies",
    expected_outcome="Traveler understands the rebooking process and any associated fees",
    user_description="First-time flyer, unfamiliar with airline procedures",
    turns=[
        Turn(role="assistant", content="Welcome to SkyTravel! How can I help you with your trip today?"),
    ]
)]

test_cases = simulator.simulate(
    conversational_goldens=goldens,
    max_user_simulations=5    
)

print(f"\nSimulated {len(test_cases)} conversations.")
for i, conv in enumerate(test_cases):
    print(f"\n--- Conversation {i+1} ({len(conv.turns)} turns) ---")
    for turn in conv.turns:
        preview = turn.content[:120].replace("\n", " ")
        print(f"  [{turn.role}] {preview}..." if len(turn.content) > 120 else f"  [{turn.role}] {preview}")

### Defining Simulation Lifecycle Hooks

In some cases, you might prefer to handle results as each conversation wraps up instead of waiting for the entire batch of cases to finish. To enable this, use the `on_simulation_complete` hook. The following hook prints basic completion message after each simulation completion and warns users in a case of conversation legnth overflow.

The hook parameters are:

1. `test_case` the completed ConversationalTestCase
2. `index` the index of the corresponding golden (ordering is preserved)

If `async_mode=True` is defined, conversations may complete out of order. You can use index parameters to track the correspondence between goldens and tests.

### Running Simulations in Parallel

Running simulations in parallel allow to significantly decrease simulation latency. By default, async_mode=True and the simulator runs conversations concurrently. 

In [ ]:
from deepeval.test_case import ConversationalTestCase

goldens = [(ConversationalGolden(
            scenario="Traveler searching for a round-trip flight from Boston to Rome",
            expected_outcome="User compares options and books a flight",
            user_description="Easygoing traveler with flexible dates who is happy to take recommendations"
            )),
            (ConversationalGolden(
            scenario="Traveler searching for a round-trip flight from Boston to Rome",
            expected_outcome="User compares options and books a flight",
            user_description="Meticulous planner who asks about every fare detail, baggage policy, and layover duration before committing"
            ))]

def handle_complete(test_case: ConversationalTestCase, index: int):
    print(f"Conversation {index}: {len(test_case.turns)} turns")
    if len(test_case.turns) >= 3:
        print(f" Attention: Long conversation can indicate a resolution failure")

test_cases = simulator.simulate(
    conversational_goldens=goldens,
    max_user_simulations=5,  # max turns per conversation
    on_simulation_complete=handle_complete
)

## Best Practices for Simulation

| Practice | Guidance |
|----------|----------|
| Write specific user descriptions | A vague persona like "a customer" produces <br>generic conversations. A specific one like <br>"impatient user who repeats questions when <br>confused" generates more realistic and <br>challenging interactions. |
| Cover the full spectrum of behavior | Pair the same scenario with cooperative, <br>frustrated, confused, and adversarial personas <br>to test how your application adapts. |
| Test edge cases explicitly | Do not rely on organic variation to surface <br>boundary conditions. Add scenarios for invalid <br>inputs, off-topic requests, and mid-conversation <br>topic switches. |
| Start short, then increase depth | Use 3–5 turns for quick smoke tests during <br>development. Increase to 15–20 turns for <br>production benchmarks that stress context <br>retention and coherence. |
| Use lifecycle hooks for monitoring | Attach an `on_simulation_complete` callback <br>to flag long conversations (potential resolution <br>failures) or log results as they finish. |
| Version your golden scenarios | Store `ConversationalGolden` definitions in <br>source control so evaluation criteria evolve <br>with your application. |
| Run simulations in CI/CD | Execute the same scenario suite on every code <br>change to catch conversational regressions <br>before they reach production. |


# Next Steps

- Continue to **`04-11-04-deepeval-simulation.ipynb`** to learn how to use Deep Eval to evaluate multi-turn conversations
- Or skip to **`04-11-05-strands-evaluators.ipynb`** to learn how to use Strands Eval to evaluate multi-turn conversations